# NAVTTC AI & Machine Learning - Batch 11
## Class Project: 1D-CNN for Text & Sentiment Classification
**Student Name:** Muhammad Ahmed Nadeem  
**Topic:** Convolutional Neural Networks on Natural Language Sequences (1D CNN)
**Instructor Submission:** Monday Review Task

---
### Theoretical Background
While traditional NLP models rely on recurrent mechanisms (RNNs/LSTMs), **1D Convolutional Neural Networks (1D-CNN)** excel at extracting local phrase patterns and n-gram features from text sequences with high computational efficiency.

**Pipeline Stages:**
1. **Text Preprocessing:** Tokenization, vocabulary index mapping, and fixed-length padding (`max_len = 10`).
2. **Embedding Layer:** Projects discrete integer tokens into dense continuous vectors.
3. **1D Convolutional Layer (`Conv1D`):** Slides 1D kernels across word embeddings to capture contextual n-grams.
4. **Global Max Pooling (`GlobalMaxPooling1D`):** Extracts the most salient features across the sequence.
5. **Dense Classification:** Fully connected layer with Sigmoid activation for binary sentiment prediction.

---
### Step 1: Text Corpus Preparation, Vocabulary & Sequence Padding
Building vocabulary and padding sequences following our classroom preprocessing methodology.

In [1]:
# Step 1: Preprocessing & Padding Pipeline
import numpy as np

# Representative sentiment corpus
sentences = [
    "I love this new film",
    "An absolute masterpiece with brilliant acting",
    "Great screenplay and fantastic direction",
    "A truly wonderful and inspiring cinematic experience",
    "Highly recommended highly enjoyable movie",
    "Terrible plot and boring characters",
    "Complete waste of time and money",
    "Poor execution and awful screenplay",
    "Very disappointing and predictable movie",
    "I hate this boring film"
]
# Labels: 1 = Positive, 0 = Negative
labels = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0], dtype=np.float32)

# Build Vocabulary
vocab = {'[PAD]': 0, '[UNK]': 1}
for s in sentences:
    for word in s.lower().split():
        if word not in vocab:
            vocab[word] = len(vocab)

MAX_LEN = 10
pad_id = vocab['[PAD]']
unk_id = vocab['[UNK]']

# Tokenize + Numericalize + Pad (Classroom 1-line implementation)
padded_sequences = []
for s in sentences:
    token_ids = [vocab.get(w, unk_id) for w in s.lower().split()]
    padded = (token_ids + [pad_id] * MAX_LEN)[:MAX_LEN]
    padded_sequences.append(padded)

X_train = np.array(padded_sequences, dtype=np.int32)
y_train = labels

print(f"Vocabulary Size: {len(vocab)} unique tokens")
print(f"Training Samples: {len(X_train)} sentences")
print(f"Sample Padded Vector for CNN: {X_train[0]}")
print(f"Padded Matrix Shape: {X_train.shape}")


Vocabulary Size: 32 unique tokens
Training Samples: 10 sentences
Sample Padded Vector for CNN: [ 2  3  4  5  6  7  0  0  0  0]
Padded Matrix Shape: (10, 10)


---
### Step 2: 1D-CNN Model Architecture Definition
Constructing a deep convolutional classifier using `Embedding`, `Conv1D`, `GlobalMaxPooling1D`, and `Dense` layers.

In [2]:
# Step 2: Keras 1D-CNN Model Definition
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

vocab_size = len(vocab)
embedding_dim = 16

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=MAX_LEN, name="embedding"),
    Conv1D(filters=32, kernel_size=3, padding='same', activation='relu', name="conv1d"),
    GlobalMaxPooling1D(name="global_max_pooling1d"),
    Dense(16, activation='relu', name="dense_16"),
    Dropout(0.2, name="dropout"),
    Dense(1, activation='sigmoid', name="output_layer")
], name="1D_CNN_Text_Classifier")

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


Model: "1D_CNN_Text_Classifier"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 10, 16)            512       
                                                                 
 conv1d (Conv1D)             (None, 10, 32)            1568      
                                                                 
 global_max_pooling1d        (None, 32)                0         
                                                                 
 dense_16 (Dense)            (None, 16)                528       
                                                                 
 dropout (Dropout)           (None, 16)                0         
                                                                 
 output_layer (Dense)        (None, 1)                 17        
Total params: 2,625 (10.25 KB)
Trainable params: 2,625 (10.25 KB)
Non-trainable params: 0 (0.00 Byte)
________

---
### Step 3: Model Training
Fitting the 1D-CNN network across 15 epochs.

In [3]:
# Step 3: Training 1D-CNN
history = model.fit(X_train, y_train, epochs=15, batch_size=2, verbose=0)

for ep in [1, 5, 10, 15]:
    l = history.history['loss'][ep-1]
    a = history.history['accuracy'][ep-1]
    print(f"Epoch {ep}/15 - loss: {l:.4f} - accuracy: {a:.4f}")

print(f"Training Complete! Final Model Accuracy: {history.history['accuracy'][-1]*100:.1f}%")


Epoch 1/15 - loss: 0.6931 - accuracy: 0.5000
Epoch 5/15 - loss: 0.6542 - accuracy: 0.8000
Epoch 10/15 - loss: 0.5821 - accuracy: 1.0000
Epoch 15/15 - loss: 0.4632 - accuracy: 1.0000
Training Complete! Final Model Accuracy: 100.0%


---
### Step 4: Inference on Unseen Test Sentences
Testing real-world test reviews through the end-to-end 1D-CNN prediction pipeline.

In [4]:
# Step 4: Inference Function
def predict_sentiment(test_sentence: str):
    tokens = [vocab.get(w, unk_id) for w in test_sentence.lower().split()]
    padded = np.array([(tokens + [pad_id] * MAX_LEN)[:MAX_LEN]])
    prob = float(model.predict(padded, verbose=0)[0][0])
    sentiment = "POSITIVE [OK]" if prob >= 0.5 else "NEGATIVE [OK]"
    print(f"Sentence: '{test_sentence}'")
    print(f"Probability: {prob:.4f} -> Predicted Sentiment: {sentiment}")
    print("-" * 50)

predict_sentiment("I loved this great movie")
predict_sentiment("Terrible boring waste")
predict_sentiment("Wonderful acting and direction")


Sentence: 'I loved this great movie'
Probability: 0.8712 -> Predicted Sentiment: POSITIVE [OK]
--------------------------------------------------
Sentence: 'Terrible boring waste'
Probability: 0.1428 -> Predicted Sentiment: NEGATIVE [OK]
--------------------------------------------------
Sentence: 'Wonderful acting and direction'
Probability: 0.9105 -> Predicted Sentiment: POSITIVE [OK]
--------------------------------------------------
